In [14]:
# Uncomment the lines below and run the installations
#!pip install torch
#!pip install torchvision
#!pip install gradio

In [1]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import gradio as gr
from sklearn.metrics import precision_score, recall_score, accuracy_score

In [2]:
# Calculating the mean and standard deviation of the data set
def load_images_from_folder(folder, target_size=(256, 256)):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, target_size)
            images.append(img)
    return images

def compute_mean_std(images):
    stacked_images = np.stack(images, axis=0)
    mean = np.mean(stacked_images)
    std = np.std(stacked_images)
    return mean, std

folder_path = r'C:\Users\Khomp\Desktop\Projetos SBC 2025\Official\Health-projects\ResNet_Trained_with_synthetic_chest_eray_imgs\Chest-xray-classification\checandoMEDIAeSD'
images = load_images_from_folder(folder_path)
mean, std = compute_mean_std(images)
print(f'Mean: {mean}')
print(f'Standard Deviation: {std}')

Mean: 141.33216187049604
Standard Deviation: 78.7593975383086


In [3]:
# Defining data transformations for grayscale images with the computed mean and std
data_transforms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),  # Converting to grayscale
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([mean / 255], [std / 255])  # Normalizing using the computed mean and std
    ]),
    'val': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),  # Converting to grayscale
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([mean / 255], [std / 255])  # Normalizing using the computed mean and std
    ]),
}


In [4]:
# Defining the data directory
data_dir = r'C:\Users\Khomp\Desktop\Projetos SBC 2025\Official\Health-projects\ResNet_Trained_with_synthetic_chest_eray_imgs\Chest-xray-classification\dataset'

In [5]:
# Creating data loaders
image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=4, shuffle=True, num_workers=4) for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
print(dataset_sizes)
class_names = image_datasets['train'].classes
print('Classes: ', class_names)

{'train': 91, 'val': 25}
Classes:  ['synthetic chest x-ray']


In [6]:
# Loading the pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)

# Freezing all layers except the final classification layer
for name, param in model.named_parameters():
    if "fc" in name:  # Unfreezing the final classification layer
        param.requires_grad = True
    else:
        param.requires_grad = False

# Defining the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Moving the model to the GPU if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

C:\Users\Khomp\USPesalqMBA\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Khomp\USPesalqMBA\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# Training loop
num_epochs = 2
for epoch in range(num_epochs):
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()
        running_loss = 0.0
        running_corrects = 0
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            # Converting grayscale images to RGB
            if inputs.shape[1] == 1:
                inputs = inputs.repeat(1, 3, 1, 1)
            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        epoch_loss = running_loss / dataset_sizes[phase]
        epoch_acc = running_corrects.double() / dataset_sizes[phase]
        print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
print("Training complete!")

train Loss: 1.7678 Acc: 0.7692
val Loss: 0.0000 Acc: 1.0000
train Loss: 0.0000 Acc: 1.0000
val Loss: 0.0000 Acc: 1.0000
Training complete!


In [8]:
# Evaluation
def evaluate_model(model, dataloaders, dataset_sizes, device):
    model.eval()
    for phase in ['train', 'val']:
        all_preds = []
        all_labels = []
        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)
            # Converting grayscale images to RGB
            if inputs.shape[1] == 1:
                inputs = inputs.repeat(1, 3, 1, 1)
            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        # Converting lists to NumPy arrays
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')
        print(f'{phase} Accuracy: {accuracy:.4f}')
        print(f'{phase} Precision: {precision:.4f}')
        print(f'{phase} Recall: {recall:.4f} \n')

# Calling the function after training the model
evaluate_model(model, dataloaders, dataset_sizes, device)

train Accuracy: 1.0000
train Precision: 1.0000
train Recall: 1.0000 

val Accuracy: 1.0000
val Precision: 1.0000
val Recall: 1.0000 



In [9]:
# Saving the model
torch.save(model.state_dict(), 'chest_xray_classification_model.pth')

In [10]:
# Loading the saved model
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 1000)  # Adjusting to match the original model's output units
model.load_state_dict(torch.load('chest_xray_classification_model.pth'))
model.eval()

# Creating a new model with the correct final layer
new_model = models.resnet50(pretrained=True)
new_model.fc = nn.Linear(new_model.fc.in_features, 2)  # Adjusting to match the desired output units
# Copying the weights and biases from the loaded model to the new model
new_model.fc.weight.data = model.fc.weight.data[0:2]  # Copying only the first 2 output units
new_model.fc.bias.data = model.fc.bias.data[0:2]

C:\Users\Khomp\USPesalqMBA\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Khomp\USPesalqMBA\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
# Defining the preprocessing function
def preprocess_image(image):
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    input_tensor = preprocess(image)
    input_batch = input_tensor.unsqueeze(0)  # Add a batch dimension
    return input_batch

# Defining the prediction function
def predict(image):
    input_batch = preprocess_image(image)
    # Check if CUDA is available and use it if possible
    if torch.cuda.is_available():
        input_batch = input_batch.to('cuda')
        new_model.to('cuda')
    # Performing inference
    with torch.no_grad():
        output = new_model(input_batch)
        # Getting the predicted class
        _, predicted_class = output.max(1)
    class_names = ['real chest x-ray', 'synthetic chest x-ray']  # Updated class names
    predicted_class_name = class_names[predicted_class.item()]
    return predicted_class_name, image

# Creating the Gradio interface
def gradio_predict(image):
    predicted_class_name, image = predict(image)
    displayed_image = display_image_with_prediction(predicted_class_name, image)
    return predicted_class_name, displayed_image

iface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="pil"),
    outputs=["text", "image"],
    title="Chest X-ray Classification",
    description="Upload a chest x-ray image to classify it as real or synthetic."
)

# Launching the interface
iface.launch()

# Defining a function to display the image with the predicted class name
def display_image_with_prediction(predicted_class_name, image):
    image_np = np.array(image)
    plt.imshow(image_np, cmap='gray')
    plt.axis('off')
    plt.text(10, 10, f'Predicted: {predicted_class_name}', fontsize=12, color='white', backgroundcolor='red', weight='bold')
    plt.savefig("predicted_image.png")  # Save the plot as an image file
    return Image.open("predicted_image.png")

* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.
